# SmolLM3 Model

In [ ]:
"""! @brief Simple/Complex Text Classification with Hugging Face Tb SmolLM3"""
# @file SmolLM3.ipynb
#
# @mainpage SmolLM3
#
# @section description_main Description
# This project evaluates and compares the simple/complex text classification SmolLM3 with and without few shots.
# The idea is to compare the performance of the models in terms of accuracy with and without few shots.
#
# @section libraries_main Libraries
# - gc (https://docs.python.org/3/library/gc.html)
#   - Used to release unused GPU memory
#
# @section imports_main Imports
# - LLMs (https://github.com/Luco1421/TPs_IA/blob/master/TP2/LLMs.py)
#   - Used to read the dataset and some test LLM's functions
#
# @section models_main Models
# - SmolLM3 (https://huggingface.co/HuggingFaceTB/SmolLM3-3B)
#   - Model to evaluate
#
# @section authors_main Authors
# - Alejandro Cerdas
# - Kener Castillo
# - Pablo Perez

## Imports

In [ ]:
# @brief Initialization of dataset and LLM utility objects.

# Import
import gc

# Import all utilities and classes from the LLMs module
from LLMs import *

# Load dataset from Excel file
dataset = ReadDataset("FEINA_1.xlsx").read()

# Create utility instance for LLM testing
llm_utils = TestLLMUtils()

## Charge model

In [ ]:
# @brief GPU cleanup and SmolLM3 model initialization.

# Release unused GPU memory resources
clean_gpu()

# Load pretrained SmolLM3 language model
smolLM3 = charge_model("HuggingFaceTB/SmolLM3-3B")

## Function to request SmolLM3's answer

In [ ]:
# @brief Generates responses using the SmolLM3 language model.

def chat_smolLM3(text: str, prompt: str):
    """! Generates a response using the SmolLM3 model.

    @details
    This function formats a conversation using the chat
    template of the tokenizer, tokenizes the input,
    performs text generation with the language model,
    decodes the generated response, and extracts
    the final processed result.

    @param text Input text.
    @param prompt System prompt used to guide generation.

    @return Processed generated response.
    """

    # Define conversation messages
    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": text},
    ]

    # Apply tokenizer chat template
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    # Tokenize formatted prompt
    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    # Disable gradient computation during inference
    with torch.inference_mode():

        # Generate model response
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            temperature=0.1,
            top_p=0.1,
            do_sample=True
        )

    # Decode generated response tokens
    response_text = tokenizer.decode(outputs[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)

    # Release unused GPU memory
    clean_gpu()

    # Return processed response
    return extract_results(response_text)

# Extract tokenizer and model from the loaded SmolLM3 tuple
tokenizer, model = smolLM3

### Example of use with without shots

In [ ]:
#@brief Executes experiments with SmolLM3.

# Generate a response using predefined samples and context
chat_smolLM3(SAMPLE_1 + SAMPLE_2 + SAMPLE_3, CONTEXT + TEXT_BEGINNER)

### Example of use with with shots

In [ ]:
# @brief Executes few-shot inference experiments with SmolLM3.

# Create batching utility instance
batcher = Batcher()

# Evaluate the model with different shot configurations
for i in [2,4,7]:

    # Generate few-shot examples from the dataset
    shot = batcher.set_shots(dataset.corpus[-i:],dataset.labels[-i:])

    # Generate response using few-shot prompting
    chat_smolLM3(SAMPLE_1 + SAMPLE_2 + SAMPLE_3, CONTEXT + SHOTS_BEGINNER + shot + TEXT_BEGINNER)

# Results

# Test without Few Shots

In [ ]:
# @brief Evaluates SmolLM3 performance without few-shot examples.

# Run all dataset evaluation for SmolLM3 without few-shot
llm_utils.test_without_shots("SmolLM3", chat_smolLM3, dataset)

# Test with Few Shots

In [18]:
# @brief Evaluates SmolLM3 performance with few-shot examples.

# Run all dataset evaluation for SmolLM3 with 2,4 and 7 shots
llm_utils.test_with_shots("SmolLM3", chat_smolLM3, dataset, [2, 4, 7])

Hugging Face TB with 2 shots - Accuracy: 0.49706518381217174
Hugging Face TB with 4 shots - Accuracy: 0.4851085310449268
Hugging Face TB with 7 shots - Accuracy: 0.48815086782376504


# 30 Runs with 80/20 partition

In [7]:
# @brief Evaluates SmolLM3 using 30 different random partitions (splits) of the dataset

# Run full evaluation for SmolLM3
llm_utils.test_all_LLM("SmolLM3", chat_smolLM3, dataset, [2])

Hugging Face TB without shots: average = 0.4976, std = 0.0141
Hugging Face TB with 2 shots - average = 0.4983, std = 0.0142
Hugging Face TB with 4 shots - average = 0.4988, std = 0.0152
Hugging Face TB with 7 shots - average = 0.4999, std = 0.0122
--------------------------------------------------


{0: [0.488905325443787,
  0.5150602409638554,
  0.48628048780487804,
  0.4809451219512195,
  0.5129573170731707,
  0.4880774962742176,
  0.5069801616458487,
  0.4958615500376223,
  0.50187265917603,
  0.5143072289156626,
  0.5038402457757296,
  0.48153730218538054,
  0.5195783132530121,
  0.5123226288274833,
  0.4954819277108434,
  0.4771341463414634,
  0.49776119402985075,
  0.5193452380952381,
  0.48134044173648133,
  0.4713855421686747,
  0.49255952380952384,
  0.5003717472118959,
  0.5,
  0.5191873589164786,
  0.4915514592933948,
  0.5163249810174639,
  0.4992378048780488,
  0.49276466108149275,
  0.48698884758364314,
  0.4784905660377359],
 2: [0.4948224852071006,
  0.49623493975903615,
  0.5022865853658537,
  0.504950495049505,
  0.4969512195121951,
  0.5216095380029806,
  0.4996326230712711,
  0.510158013544018,
  0.49887640449438203,
  0.5015060240963856,
  0.5084485407066052,
  0.4777694046721929,
  0.4894578313253012,
  0.5093353248693054,
  0.49849397590361444,
  0.463414634

In [8]:
# @brief Releases GPU memory and clears model-related variables.

# Clean GPU cached memory
clean_gpu()

# Delete model, tokenizer, and inference function references
del smolLM3, chat_smolLM3, tokenizer, model

# Force Python garbage collector to release memory
gc.collect()

2063